# RAILGUN+ — Step 2: Generate v2 Features and Retrain



## 1. Setup

In [ ]:
import os, sys
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')
import torch

# EDIT: point at your prebuilt LaCAM binary (same one as before)
LACAM_BIN='/kaggle/input/lacam-binary/main'
!chmod +x {LACAM_BIN}
DATA_DIR_V2='/kaggle/working/data_lacam_v2/shards'
CKPT_DIR='/kaggle/working/checkpoints_v2'
os.makedirs(DATA_DIR_V2, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

## 2. Generate v2 LaCAM data (resumable)


In [ ]:
for agents in [16, 32, 64, 96]:
    !cd /kaggle/working/railgun-plus && PYTHONPATH=src python scripts/generate_data.py \
        --out {DATA_DIR_V2} --expert lacam --lacam-bin {LACAM_BIN} \
        --features v2 \
        --instances 200 --map-size 32 --density 0.2 --agents {agents} --seed {agents} \
        --batch 20 --lacam-time-ms 10000

## 3. Train the v2 model (9 input channels)
*Note: this is a NEW model architecture (in_channels=9), so it starts fresh. Resume still works across disconnects.*

In [ ]:
from railgun_plus.models import RailgunUNet
from railgun_plus.data.dataset import ShardedMapfDataset
from railgun_plus.train import train
from railgun_plus.data.features_v2 import NUM_FEATURE_CHANNELS_V2

device='cuda' if torch.cuda.is_available() else 'cpu'
dataset=ShardedMapfDataset(DATA_DIR_V2)
print('samples:', len(dataset), '| input channels:', NUM_FEATURE_CHANNELS_V2)
model=RailgunUNet(in_channels=NUM_FEATURE_CHANNELS_V2, num_actions=5, base=64)
print('params:', f'{model.count_params()/1e6:.1f}M')
history=train(model, dataset,
              epochs=50, batch_size=8, lr=1e-3, weight_decay=1e-3,
              device=device, ckpt_dir=CKPT_DIR,
              val_fraction=0.1, early_stop_patience=5, min_delta=1e-3)

## 4. Save best.pt as a Kaggle Dataset
For use by the Step-2 eval notebook.

In [ ]:
!ls -la {CKPT_DIR}
print('Now: save', CKPT_DIR, 'as a Kaggle dataset for eval.')